# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadahannadeembaig/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [42]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page's daily performance for one client, on one report_date (grain: report_date + client_hash_id + content_hash_id). Time window used for development: month = '2026-03' (a mid-panel month). The final month (June 2026, the _sample file) is reserved as a sealed test month and excluded from development, since it's the natural outcome window for any label.

In [43]:
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │  cnt  │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature columns (knowable at decision time): gsc_impressions, gsc_clicks, gsc_sum_position, gsc_data_available, ga4_data_available

Label/proxy column: a decline signal derived from gsc_clicks trend over time, used to rank refresh priority.

Context columns (identifiers, not features): report_date, client_hash_id, content_hash_id, month

Excluded: the final month (2026-06 / the _sample file), since it's the outcome window and using it would leak future information. Also excluded: AI-referral columns (ai_chatgpt, ai_perplexity, ai_gemini, etc.) since they're outside my lane's scope for this task.

In [44]:
con.sql(f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as earliest, MAX(report_date) as latest
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

┌────────────┬────────────┬────────────┐
│ total_rows │  earliest  │   latest   │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [45]:
con.sql(f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘

Grain check confirmed no duplicate rows for the same date/client/content combination — zero rows returned. The month=2026-03 slice has 9,841,378 total rows, spanning 2026-03-01 to 2026-03-31. Of those, 3,611,061 rows have gsc_data_available IS TRUE — the remaining rows are missing GSC data for that client/day and would need to be excluded or imputed before modeling.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [46]:
features_df = con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_data_available
0,content_b7e512995f79d5a6,2026-03-01,20,0,67,True
1,content_05597932fe4da067,2026-03-01,1,0,0,True
2,content_7a105f548d9c6916,2026-03-01,125,1,616,True
3,content_905aa32a0230694e,2026-03-01,7,0,28,True
4,content_a3ea9792f793ec72,2026-03-01,11,0,25,True
5,content_36c36abc7650d7af,2026-03-01,239,1,1756,True
6,content_a7da352b73b02668,2026-03-01,191,0,1496,True
7,content_05434271b257bb68,2026-03-01,55,0,180,True
8,content_d056587ff7faca0c,2026-03-01,77,0,434,True
9,content_bfd1e41c2af250c8,2026-03-01,2,0,9,True


- gsc_impressions: available at decision moment because Search Console reports it same-day.
- gsc_clicks: available at decision moment because it's logged as it happens, same-day.
- gsc_sum_position: available because it's part of the same daily GSC report.
- gsc_data_available: available because it's a same-day data-quality flag, not a future signal.
- report_date itself (used to compute recency/staleness): always knowable, since it's just a date.

In [47]:
import pandas as pd

# Step 1: an honest, modest signal (no leakage)
features_df['ctr_proxy'] = features_df['gsc_clicks'] / features_df['gsc_impressions'].replace(0, 1)
honest_score = features_df['ctr_proxy'].corr(features_df['gsc_sum_position'])
print("Honest score (should be modest):", honest_score)

# Step 2: ON PURPOSE add a leaky column (derived directly from what we're trying to predict)
features_df['LEAKY_column'] = features_df['gsc_clicks']
leaky_score = features_df['LEAKY_column'].corr(features_df['gsc_clicks'])
print("Leaky score (suspiciously near-perfect):", leaky_score)

# Step 3: remove the leak, keep the honest number
features_df = features_df.drop(columns=['LEAKY_column'])
print("Leaky column removed. Honest score stands at:", honest_score)

Honest score (should be modest): 0.0063226206446219645
Leaky score (suspiciously near-perfect): 1.0
Leaky column removed. Honest score stands at: 0.0063226206446219645


When I added a column derived directly from the same signal I was trying to predict, my score jumped to a near-perfect correlation — a clear sign of leakage, not a genuinely good model. I removed it and kept the honest, modest score, which reflects what can actually be predicted using only information available before the

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation of this slice: the panel is unbalanced — different clients have different amounts of GSC/GA4 history (see dim_clients.gsc_data_start / ga4_data_start), so early rows for some clients may simply be missing history rather than reflecting real zero activity. This means comparisons across clients within the same month aren't perfectly apples-to-apples.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.